# 3장 임베딩: 정수를 벡터로 (실습)

교재 `docs/book/03-embedding.md` 와 함께 본다. 이 노트북에서 하는 것:

1. 토큰 id → 벡터 룩업이 "원-핫 × 행렬"과 같음을 확인하고 shape `(B, T, C)` 를 만든다
2. 랜덤 벡터는 아무 뜻이 없음을 보고, **세어서** 만든 벡터(동시출현 + SVD)가 이웃 토큰을 찾아내는 것을 본다
3. 위치 임베딩, 왜 필요한지, sin/cos 방식과 학습 방식
4. `GPTEmbedding` 으로 6장 GPT 의 입구를 조립하고 파라미터 수를 센다

> 전체 실행 약 30초.

## 1. 룩업 테이블 = 원-핫 × 행렬

In [ ]:
import torch
import torch.nn.functional as F

from shllm.config import TOKENIZER_DIR, setup_cpu
from shllm.data import load_corpus
from shllm.embedding import GPTEmbedding, TokenEmbedding, one_hot_lookup
from shllm.tokenizer import BPETokenizer

setup_cpu()
tok = BPETokenizer.load(TOKENIZER_DIR / "bpe-8192.json")
V, C = tok.vocab_size, 8  # 눈으로 보려고 C 를 작게

emb = TokenEmbedding(V, C)
idx = torch.tensor([tok.encode("옛날 옛적에")])  # (B=1, T)
print("idx", tuple(idx.shape), idx.tolist())
x = emb(idx)
print("x  ", tuple(x.shape), "= (B, T, C)")
print("원-핫 경로와 같은가:", torch.allclose(x, one_hot_lookup(idx, emb.weight)))
print("토큰 '옛' 의 벡터 = weight[%d]:" % idx[0, 0], x[0, 0].detach().numpy().round(3))

**출력에서 볼 것**: `x` 의 shape 이 `(1, 5, 8)`, 배치 1, 토큰 5개, 각 토큰이 8개 숫자. `x[0, 0]` 은 `emb.weight[1059]` 행과 똑같다.

원-핫 벡터는 길이 V(8,192) 중 한 칸만 1 이라, 행렬과 곱하면 그 행이 그대로 나온다. 그러니 곱셈 대신 **행을 꺼내면** 된다. 이것이 `nn.Embedding` 의 전부다.
표의 값은 `nn.Parameter` 라 학습 대상이다. 지금은 랜덤이다.

## 2. 벡터에 뜻이 생기려면

In [ ]:
from shllm.embedding import cooccurrence_matrix, nearest, ppmi, svd_embeddings

text = load_corpus("korean-classics")
ids = torch.tensor(tok.encode(text))
freq = torch.bincount(ids, minlength=V)
N = 4000  # 흔한 토큰 4,000개만 (드문 토큰은 세어도 정보가 없다)
top = torch.argsort(freq, descending=True)[:N]
names = [tok.token_str(int(i)) for i in top]
remap = torch.full((V,), -1, dtype=torch.long)
remap[top] = torch.arange(N)
r = remap[ids]
r = r[r >= 0]  # 상위 N 밖 토큰은 뺀다 (경계가 살짝 흐트러지지만 감을 잡는 데는 충분)


def neighbors(E, word, k=6):
    i = names.index(word)
    return [(names[j], round(s, 2)) for j, s in nearest(E, i, k)]


random_E = torch.randn(N, 64)
print("랜덤 벡터에서 ' 아버지' 의 이웃:", neighbors(random_E, " 아버지"))

당연히 아무 관계도 없는 토큰들이다. 벡터에 뜻이 생기는 방법은 둘이다.

- **학습**: 다음 토큰 예측 손실이 줄어드는 방향으로 표를 조금씩 고친다 (4장부터 하는 일, 실제 GPT 방식)
- **세기**: "비슷한 문맥에 나오는 토큰은 비슷한 뜻"(분포 가설). 앞뒤 2칸 안에 같이 나온 횟수를 세어 (V, V) 표를 만들고 C 차원으로 압축한다

두 번째를 지금 해 본다. 1장의 확률표를 압축하면 벡터가 된다는 감각이 생긴다.

In [ ]:
counts = cooccurrence_matrix(r, N, window=2)  # (N, N)
print("동시출현 표", tuple(counts.shape), "| ' 아버지' 옆에 가장 많이 온 토큰:",
      [names[j] for j in torch.topk(counts[names.index(" 아버지")], 5).indices])
M = ppmi(counts)  # 흔한 토큰의 영향을 빼고
E = svd_embeddings(M, n_embd=64)  # (N, 64)
print("SVD 벡터", tuple(E.shape))
for w in (" 아버지", " 돈", " 서울", " 없다", "는", ","):
    print(f"{w!r:>8} →", neighbors(E, w))

**해 보기**: `neighbors(E, " 사랑")`, `neighbors(E, "…")` 처럼 다른 토큰을 넣어 보라 (상위 4,000 안에 있어야 한다. `names` 에 있는지 먼저 확인). `window=2` 를 5 로 바꾸면 이웃이 "문법적 역할"보다 "주제"쪽으로 기운다.

` 아버지` 옆에는 ` 어머니`·` 언니`, ` 돈` 옆에는 ` 원`·` 이천`·` 오백`, ` 서울` 옆에는 ` 동경`·` 시골`. **아무도 뜻을 가르치지 않았다**, 같은 자리에 나오는 토큰이 같은 방향의 벡터가 됐을 뿐이다.
`는` 의 이웃은 `가`·`를`·`고` 처럼 조사·어미끼리 모인다. 문법적 역할도 "문맥"이다.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

cjk = [f.name for f in font_manager.fontManager.ttflist if "CJK" in f.name]
if cjk:
    matplotlib.rcParams["font.family"] = cjk[0]

groups = {
    "가족": [" 아버지", " 어머니", " 아들", " 딸", " 아내", " 남편"],
    "돈·숫자": [" 돈", " 원", " 백", " 천", " 만", " 오백"],
    "조사·어미": ["는", "가", "를", "고", "었다", "는다"],
    "장소": [" 서울", " 시골", " 집", " 학교", " 방", " 병원"],
}
present = {g: [w for w in ws if w in names] for g, ws in groups.items()}
sel = [w for ws in present.values() for w in ws]
X = E[[names.index(w) for w in sel]]
X = X - X.mean(0)
U, S, Vt = torch.linalg.svd(X, full_matrices=False)  # 선택한 점들만 2차원으로 (PCA)
P = X @ Vt[:2].T

fig, ax = plt.subplots(figsize=(7, 5.5))
i = 0
for g, ws in present.items():
    pts = P[i : i + len(ws)]
    ax.scatter(pts[:, 0], pts[:, 1], label=g, s=40)
    for w, (px, py) in zip(ws, pts):
        ax.annotate(w.strip(), (px, py), fontsize=9, xytext=(3, 3), textcoords="offset points")
    i += len(ws)
ax.set_title("SVD 임베딩 64차원 → 2차원 (PCA)")
ax.legend()
plt.show()

64차원을 2차원으로 눌러 본 그림이라 정보가 많이 사라졌지만, 무리별로 뭉치는 경향은 보인다.
학습으로 만든 임베딩도 같은 성질을 가지며, 다음 토큰 예측이라는 목적에 맞춰 더 정교해진다. 7장 학습 후 같은 그림을 다시 그린다.

## 3. 위치 임베딩, 순서를 잃어버리지 않기

토큰 임베딩만 있으면 `[개, 가, 사람, 을, 물었다]` 와 `[사람, 이, 개, 를, 물었다]` 의 벡터 **집합**이 거의 같다.
5장 어텐션은 벡터들의 가중 합이라 순서를 모른다. 그래서 "몇 번째 자리인가"도 벡터로 만들어 **더해 준다**.

In [ ]:
from shllm.embedding import sinusoidal_positions

pe = sinusoidal_positions(n_positions=64, n_embd=64)  # (T, C)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(pe, aspect="auto", cmap="RdBu")
axes[0].set_xlabel("채널 C")
axes[0].set_ylabel("위치 T")
axes[0].set_title("sin/cos 위치 벡터 (열마다 다른 주기)")
sim = pe @ pe.T / (pe.norm(dim=1)[:, None] * pe.norm(dim=1)[None, :])
axes[1].imshow(sim, cmap="viridis")
axes[1].set_title("위치끼리의 코사인 유사도")
axes[1].set_xlabel("위치")
plt.show()

왼쪽: 앞 열은 빠르게 진동해 이웃 위치를 구분하고, 뒤 열은 느리게 변해 멀리 떨어진 위치를 구분한다.
오른쪽: 가까운 위치일수록 벡터가 비슷하다(대각선 근처가 밝다). "거리" 가 벡터에 들어 있다.

GPT-2 는 이 대신 위치마다 벡터를 **학습**한다(`LearnedPositionalEmbedding`). 단순하고 잘 되지만, 학습한 길이(`block_size`)를 넘는 문장은 처리할 수 없다. 우리도 GPT-2 방식을 쓴다.

## 4. GPTEmbedding, 6장 GPT 의 입구

In [ ]:
V, T_max, C = tok.vocab_size, 256, 256
gpt_in = GPTEmbedding(vocab_size=V, block_size=T_max, n_embd=C)
n_params = sum(p.numel() for p in gpt_in.parameters())
print(f"토큰 표 {V}×{C} = {V * C:,}  +  위치 표 {T_max}×{C} = {T_max * C:,}  →  {n_params:,} 파라미터")

batch = torch.stack([ids[i : i + 32] for i in range(0, 4 * 32, 32)])  # (B=4, T=32) 코퍼스에서 잘라낸 배치
x = gpt_in(batch)
print("batch", tuple(batch.shape), "→ x", tuple(x.shape))

# 같은 토큰이 다른 자리에 있으면 다른 벡터가 된다
same = torch.tensor([[batch[0, 0], batch[0, 0]]])
with torch.no_grad():  # 미분 이력이 필요 없는 계산, 4장에서 설명
    y = gpt_in(same)
print("같은 토큰, 자리 0 과 1 의 코사인 유사도:", round(F.cosine_similarity(y[0, 0], y[0, 1], dim=0).item(), 3))

**출력에서 볼 것**: 같은 토큰을 자리 0 과 1 에 두었을 때 코사인 유사도가 1 이 아니다. 위치 벡터가 더해져 서로 다른 벡터가 됐다. 그렇다고 0 도 아니다. 토큰 정보는 남아 있다.

파라미터 210만 개 중 위치 표는 6만 개, 임베딩의 비용은 거의 전부 **어휘 크기 V** 다 (2장의 트레이드오프).
`(B, T, C)` 가 나왔다. 여기서부터 5·6장의 어텐션과 Transformer 블록은 이 모양을 유지하며 값만 바꾼다.

## 정리

- 임베딩 = `(V, C)` 룩업 테이블. 원-핫 × 행렬과 같지만 행을 꺼낸다. 값은 파라미터라 학습된다.
- 뜻은 "같은 문맥에 나오는가"에서 온다. 세어서(PPMI+SVD) 만들어도 이웃이 잡히고, 학습하면 목적에 맞게 더 정교해진다.
- 순서는 위치 임베딩을 **더해서** 넣는다. GPT-2 는 학습형, 원조 Transformer 는 sin/cos.
- `GPTEmbedding(idx) → (B, T, C)`. 이 텐서가 모델 나머지 부분의 입력이다.

---
**다음 장**: 4장, 신경망 기초. 이 표를 "학습"한다는 것이 정확히 무엇인지, autograd 와 경사하강으로.